# 04 - DistilBERT transformer comparison

Fine-tune `distilbert-base-uncased` to predict the CFPB `Issue` from complaint narratives, then compare it with TF-IDF Logistic Regression and Linear SVM. The notebook creates a deterministic, approximately balanced sample capped at 20,000 rows. It runs both the traditional baseline and DistilBERT on the **same sampled rows and exact stratified 80/20 split** (seed 42); this means the traditional models are rerun on the balanced sample, rather than comparing against Stage 3's full-data holdout. Text preprocessing/tokenization for DistilBERT truncates or pads to 256 tokens.

**Colab setup:** select `Runtime > Change runtime type > T4 GPU` if available. The dataset is not bundled with this project; place the CFPB CSV in Google Drive and set its path below. Complaint narratives may include personal details. Only process data in Colab if your data-handling rules permit it; outputs contain example narratives and are saved in your Drive folder.

In [ ]:
# Install the NLP packages while using Colab's preinstalled CUDA-compatible PyTorch.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "scikit-learn>=1.4,<2.0",
        "transformers>=4.46,<5.0",
        "accelerate>=0.34,<2.0",
    ],
    check=True,
)

# Clone the project so the notebook uses the committed training and sampling code.
from pathlib import Path
import pandas as pd

project_root = Path("/content/Consumer-Complaint-Risk-Intelligence-Platform")
if not project_root.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/mgcoder27/Consumer-Complaint-Risk-Intelligence-Platform.git",
            str(project_root),
        ],
        check=True,
    )
sys.path.insert(0, str(project_root))

# Confirm the notebook is using the requested accelerator before long training.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. In Colab choose Runtime > Change runtime type > T4 GPU, "
        "then rerun this cell."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive so the large CFPB source file and generated outputs persist.
from google.colab import drive

drive.mount("/content/drive")

# Set these paths to your own Drive file and output folder.
input_csv = Path("/content/drive/MyDrive/cfpb/complaints.csv")
output_dir = Path("/content/drive/MyDrive/cfpb/stage4_transformer")
sample_csv = output_dir / "balanced_sample.csv"

if not input_csv.is_file():
    raise FileNotFoundError(f"Upload or copy the CFPB CSV to: {input_csv}")

# Import the two-pass sampler and the paired model runner from the project.
import json
from src.balanced_sample import load_balanced_sample
from src.transformer_baseline import train_and_compare

# Build a reproducible balanced sample using only narratives and Issue labels.
sample = load_balanced_sample(
    input_csv,
    target_size=20_000,
    random_seed=42,
    chunksize=100_000,
)
sample_csv.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(sample_csv, index=False, encoding="utf-8-sig")
(output_dir / "balanced_sample.manifest.json").write_text(
    json.dumps(sample.attrs["manifest"], indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Sample rows: {len(sample):,}")
sample_manifest = sample.attrs["manifest"]
print(f"Issue classes included: {sample['Issue'].nunique():,}")
print(f"Rows per included class: {sample_manifest['rows_per_selected_class']:,}")
print(f"Classes excluded by the 20,000-row cap/stratification rule: {len(sample_manifest['excluded_classes']):,}")
display(sample["Issue"].value_counts().rename("sample_rows").to_frame())
if sample_manifest["excluded_classes"]:
    display(pd.Series(sample_manifest["excluded_classes"], name="source_rows_excluded").sort_values(ascending=False).to_frame())

## Train and compare

The baseline evaluation creates the seeded stratified split first. Its train/test row positions are passed directly to DistilBERT, preventing split drift. No TF-IDF vectorizer is fitted on test data; the transformer tokenizer is pretrained, while supervised fine-tuning uses only training rows. Class balance is created before splitting and is shared across all three models. DistilBERT trains for a fixed two epochs without evaluating or selecting checkpoints on the test set; the holdout is used only once for final prediction and reporting.

In [ ]:
# Run both baseline models and the transformer using identical sample and test rows.
comparison_results = train_and_compare(
    sample,
    output_dir=output_dir,
    test_size=0.2,
    random_seed=42,
    max_length=256,
    model_name="distilbert-base-uncased",
)

# Display the requested macro-F1 comparison table.
comparison_table = pd.DataFrame(comparison_results["comparison_table"])
display(comparison_table.sort_values("macro_f1", ascending=False).reset_index(drop=True))

In [ ]:
# Show the generated counts before inspecting individual paired disagreements.
import json

print(json.dumps(comparison_results["paired_error_analysis"], indent=2))

# Read only paired disagreements; limit to five examples per direction.
paired_examples = pd.read_csv(output_dir / "paired_error_examples.csv")
display(paired_examples)

# Inspect the Transformer-versus-best-baseline class reports after training.
display(
    pd.DataFrame(comparison_results["transformer_classification_report"])
    .T
    .loc[comparison_results["label_order"]]
    [["precision", "recall", "f1-score", "support"]]
    .sort_values("f1-score")
    .head(10)
)

## Reading the results

Compare macro-F1 across the three models; it weights each issue equally. The winning model on this holdout is not necessarily the best production choice: inspect the lowest per-class F1 values, support, and paired errors as well. Transformer-only wins may reflect contextual cues that are not captured by unigram/bigram TF-IDF; baseline-only wins may reflect short, highly distinctive phrases or limited transformer training. These are hypotheses to check against the displayed narratives and confusion matrix, not conclusions from model scores alone. The CSV examples include complaint text, so keep them in approved private storage.

The run saves a JSON report, macro-F1 comparison table, per-model test predictions, confusion matrices, paired error examples, and the best transformer checkpoint beneath `output_dir`. Actual metrics and weakest examples are generated only after the notebook runs on the chosen CFPB snapshot and GPU.